# Project 14 · Local-First Support Assistant and Ticket Router (Starter notebook)

**Module 14 · Local Models & Fine-Tuning** · Part 2 (Pro) · elective

| | |
|---|---|
| ⏱️ Estimated time | 4–6 hours (plus the 14.04 Colab run if you haven't done it) |
| 📋 Prerequisites | Lessons 14.01–14.04 and Project 06 (the cited knowledge assistant) |
| 🔑 Keys | none offline · optional `GROQ_API_KEY` (hosted comparison) · optional Ollama with `qwen3:4b` and your `orbita-router` |
| 💸 Cost | $0 offline; live: about 6–8K Groq tokens (16 questions); local calls are free |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |
| 📄 Spec | [project/README.md](README.md) |

**The job:** Orbita Cloud's support lead asks two questions before anything touches customer tickets: *can the documentation assistant run on our own laptops?*, and *which ticket router should we deploy, and was fine-tuning worth it?* You answer both with measured numbers on held-out data, paired comparisons, a latency budget and a privacy column, and write the recommendation.

This is the **starter**. Every `# TODO` marks work for you. The notebook runs top to bottom even before you finish: unfinished parts show ❌ in the checks instead of crashing. Stuck? Each TODO has a collapsed hint; the [reference solution](solutions/14-project-local-first-router-solution.ipynb) is the last resort.

## Requirements (from the spec)

1. **Assistant:** Project 06's questions answered by a local reader (Ollama, or the recorded run) and at least one other reader on the same context; accuracy with 95% CIs, seconds per question, cost, privacy, and a paired comparison.
2. **Router:** every router from 14.03–14.04 (baseline, few-shot local, few-shot hosted, fine-tuned) on the same 234 held-out tickets; accuracy CI, macro-F1, Spanish, latency, cost, privacy.
3. **Decision:** paired tests between tuned and few-shot models, and a recommendation under Orbita's constraints (tickets stay local, ≤ 0.5 s per ticket), saying whether fine-tuning was worth it.
4. **Report:** Markdown for the support lead.

| TODO | Where | What |
|---|---|---|
| 1 | Step 2 | `score_reader(answers)`: right answers and correct refusals |
| 2 | Step 3 | `router_row(...)`: one router's quality, latency, cost and privacy |
| 3 | Step 3 | `paired_rows(pairs)`: paired tests on shared tickets |
| 4 | Step 4 | `recommend(rows, pairs)`: the router to deploy, and why |
| 5 | Step 5 | `write_report(path, evidence)` |

---
## Step 0 · Setup

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/14-local-models-and-fine-tuning/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
LANGSMITH_PROJECT = 'm14-project'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config, look for a local Ollama.
import os

from mastery.config import env_status, is_set, load_env, load_settings
from mastery.local import ollama_status

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_HF_TOKEN = is_set("HF_TOKEN")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m14-<lesson>: one tracing project per lesson
OLLAMA = ollama_status(timeout=3.0)                              # never raises; 3 s if nothing is listening
HAVE_OLLAMA = OLLAMA.running

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast, "(Groq)")
print("local     :", settings.model_local, "| embeddings:", settings.model_local_embed)
print("ollama    :", f"{OLLAMA.version} at {OLLAMA.base_url}, models: {OLLAMA.names() or 'none pulled'}" if HAVE_OLLAMA
      else f"not running at {OLLAMA.base_url}")
if not HAVE_OLLAMA:
    print("\n⚠️ No Ollama server: 🖥️ local cells will print ⏭️ Skipped. Start the Ollama app (or `ollama serve`) and re-run.")
if not HAVE_GROQ_KEY:
    print("⚠️ No GROQ_API_KEY: 💸 Groq cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# Preview models warn once per model build; the lesson says so in Markdown already.
from mastery.models import PreviewModelWarning
warnings.filterwarnings("once", category=PreviewModelWarning)

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

---
## Step 1 · The inputs

16 questions from the Project 06 test set (12 the documentation answers, 4 it doesn't), the same BM25 index for every reader, and the router's 234 test tickets from 14.03.

In [ ]:
# The two experiments' inputs: 16 golden RAG questions (Project 06) and the 234-ticket router test set (14.03).
import json
import re
import time

from mastery.evals.stats import paired_compare, wilson_interval
from mastery.finetune import (QUEUES, RECORDED_DIR, LogReg, NaiveBayes, accuracy, by_split, correct_map, load_predictions, load_tickets,
                              macro_f1)
from mastery.pricing import estimate_cost
from mastery.rag import extractive_answer, format_context, load_corpus, split_documents
from mastery.rag.retrieve import BM25

OUT_DIR = REPO_ROOT / "data" / "outputs" / "m14"            # git-ignored
OUT_DIR.mkdir(parents=True, exist_ok=True)
rows = [json.loads(line) for line in (REPO_ROOT / "data" / "samples" / "rag" / "project06-test-set.jsonl").read_text(encoding="utf-8").splitlines()]
GOLDEN = [r for r in rows if r["in_corpus"]][::4][:12] + [r for r in rows if not r["in_corpus"]][:4]
index = BM25(split_documents(load_corpus(REPO_ROOT / "data" / "samples" / "rag" / "corpus"), preset="markdown",
                             chunk_size=800, chunk_overlap=100))
PROMPT = ("Answer the question in one short sentence using only the sources. If they don't answer it, say you don't know.\n\n"
          "{context}\n\nQuestion: {question}")
DECLINE = re.compile(r"don.t know|do not know|not (?:in|mentioned|specif|provided|stated)|no information|do not (?:say|specify|mention)", re.I)

SPLIT = by_split(load_tickets())
TRAIN, TEST = SPLIT["train"], SPLIT["test"]
GOLD = {t["id"]: t["queue"] for t in TEST}
LATENCY_BUDGET_S = 0.5     # ← the support lead's rule: a ticket is routed within half a second on a support laptop
print(f"{len(GOLDEN)} golden questions ({sum(r['in_corpus'] for r in GOLDEN)} answerable) | {len(TEST)} router test tickets")

**You should now see** `16 golden questions (12 answerable) | 234 router test tickets`.

---
## Step 2 · The assistant: local vs the rest

> 🖥️ Runs **on your computer** through Ollama: no key, no cost, nothing leaves the machine. On a **CPU-only** laptop (no NVIDIA GPU) expect 10–60 s for the first call (the model loads into RAM) and roughly 5–15 tokens per second for a 4B model afterwards; the course gives local calls a 300 s timeout. Without a running Ollama it prints `⏭️ Skipped`.

**TODO 1** is in the cell below.

> 💡 **Hint:** Loop over `GOLDEN`. For an answerable question (`in_corpus`), it's right when `answer_contains` appears in the answer (ignore case). For the four out-of-corpus questions it's right only when the reader **declined**: `DECLINE.search(text)`. A missing answer counts as wrong.

<details>
<summary>Still stuck? Show one way to write it</summary>

```python
def score_reader(answers: dict) -> dict:
    """{question id: 1 if right else 0} over GOLDEN."""
    scores = {}
    for q in GOLDEN:
        text = answers.get(q["id"], "") or ""
        if q["in_corpus"]:
            scores[q["id"]] = int(q["answer_contains"].casefold() in text.casefold())
        else:
            scores[q["id"]] = int(bool(DECLINE.search(text)))
    return scores
```

</details>

In [ ]:
def score_reader(answers: dict) -> dict:
    """{question id: 1 if right else 0} over GOLDEN."""
    # TODO 1: answerable → the answer contains answer_contains; not answerable → the reader declined (DECLINE).
    return {q["id"]: 0 for q in GOLDEN}

In [ ]:
# 🖥️💸 (HAVE_OLLAMA / HAVE_GROQ_KEY) Readers on the same 16 questions and the same BM25 context.
# Always: the offline extractive reader (Module 06) and the RECORDED local qwen3:4b run (8-core CPU, 2026-10-16).
from mastery.models import get_model

recorded = json.loads((RECORDED_DIR / "rag-qwen3-4b.json").read_text(encoding="utf-8"))
READERS = {"extractive (offline)": {"answers": {}, "seconds": {}, "local": True, "usage": None},
           "ollama:qwen3:4b (recorded)": {"answers": recorded["answers"], "seconds": recorded["seconds"], "local": True, "usage": None}}
for q in GOLDEN:
    start = time.perf_counter()
    READERS["extractive (offline)"]["answers"][q["id"]] = extractive_answer(q["question"], index.search(q["question"], k=4)).answer
    READERS["extractive (offline)"]["seconds"][q["id"]] = round(time.perf_counter() - start, 3)

live = {}
LOCAL = settings.model_local
if HAVE_OLLAMA and OLLAMA.get(LOCAL):
    live[f"ollama:{LOCAL} (this machine)"] = (get_model("fast", "ollama", temperature=0, max_tokens=200), True)
if HAVE_GROQ_KEY:
    live[f"groq:{settings.model_fast}"] = (get_model("fast", temperature=0, max_tokens=400), False)
for name, (model, is_local) in live.items():
    run = {"answers": {}, "seconds": {}, "local": is_local, "usage": {"input_tokens": 0, "output_tokens": 0}}
    for q in GOLDEN:
        start = time.perf_counter()
        try:
            reply = model.invoke(PROMPT.format(context=format_context(index.search(q["question"], k=4)), question=q["question"]))
            run["answers"][q["id"]] = reply.text
            for k in ("input_tokens", "output_tokens"):
                run["usage"][k] += (reply.usage_metadata or {}).get(k, 0)
        except Exception as error:  # noqa: BLE001 - one failed question is scored wrong, not a crash
            print(f"⚠️ {name} {q['id']}: {type(error).__name__}: {str(error)[:100]}")
            run["answers"][q["id"]] = ""
        run["seconds"][q["id"]] = round(time.perf_counter() - start, 2)
    READERS[name] = run
if not live:
    print("⏭️ Skipped live readers: no Ollama with", LOCAL, "and no GROQ_API_KEY; the extractive and RECORDED readers stand in.")
print("readers:", list(READERS))

> 💸 With `GROQ_API_KEY`: 16 calls to `gpt-oss-20b`, about 6–8K tokens. With Ollama and `qwen3:4b`: 16 local calls, about 1 minute on a CPU.

**You should now see** the readers in use: always `extractive (offline)` and `ollama:qwen3:4b (recorded)`, plus your live ones.

In [ ]:
# 🧪 The assistant table: quality with a 95% CI, seconds per question, cost, privacy.
SCORES = {name: score_reader(r["answers"]) for name, r in READERS.items()}
reader_rows = []
for name, r in READERS.items():
    right = sum(SCORES[name].values())
    usage = r["usage"]
    usd = estimate_cost(usage, f"groq:{settings.model_fast}") if usage and not r["local"] else 0.0
    reader_rows.append({"reader": name, "right": f"{right}/{len(GOLDEN)}", "accuracy [95% CI]": f"{wilson_interval(right, len(GOLDEN)):.0%}".split(" (")[0],
                        "s/question": round(sum(r["seconds"].values()) / len(GOLDEN), 2),
                        "USD/1K questions": round((usd or 0) / len(GOLDEN) * 1000, 3), "data leaves the machine": "no" if r["local"] else "yes (Groq)"})
table(reader_rows, ["reader", "right", "accuracy [95% CI]", "s/question", "USD/1K questions", "data leaves the machine"])
best_hosted = next((n for n in READERS if n.startswith("groq:")), None)
other = best_hosted or "extractive (offline)"
reader_pair = paired_compare(SCORES[other], SCORES["ollama:qwen3:4b (recorded)"], names=(other, "ollama:qwen3:4b (recorded)"))
print()
print(reader_pair)

**You should now see** (solution) the recorded qwen3:4b at about 13/16 in ~2.5 s per question with nothing leaving the machine, the extractive reader lower, Groq (live) close to the local model at a fraction of a second, and the paired result: with only 16 questions even a 4–0 win isn't significant (p ≈ 0.13), which is why the router experiment uses 234 tickets. Starter: every reader at 0/16 until TODO 1.

---
## Step 3 · The router experiment

**TODO 2** is in the cell below.

> 💡 **Hint:** Score only the ids this router labelled (`gold = {i: GOLD[i] for i in labels}`: the Groq run covers 100 tickets). Use `accuracy(labels, gold)` (an `Interval` with a Wilson CI, keep it under `"accuracy"` for later) and `macro_f1`, add `n`, Spanish accuracy and the latency/cost/privacy you were given.

<details>
<summary>Still stuck? Show one way to write it</summary>

```python
def router_row(name: str, labels: dict, seconds_per_ticket, usd_per_1k: float, local: bool) -> dict:
    """One row of the router table, scored on the test tickets this router labelled."""
    gold = {i: GOLD[i] for i in labels if i in GOLD}
    spanish = [t["id"] for t in TEST if t["lang"] == "es" and t["id"] in gold]
    return {"router": name, "n": len(gold), "accuracy": accuracy(labels, gold),
            "accuracy [95% CI]": f"{accuracy(labels, gold):.1%}".split(" (")[0],
            "macro-F1": round(macro_f1(labels, gold, QUEUES), 2),
            "Spanish": f"{sum(labels[i] == gold[i] for i in spanish) / len(spanish):.0%}" if spanish else "-",
            "s/ticket": seconds_per_ticket, "USD/1K tickets": round(usd_per_1k, 3), "local": local}
```

</details>

In [ ]:
def router_row(name: str, labels: dict, seconds_per_ticket, usd_per_1k: float, local: bool) -> dict:
    """One row of the router table, scored on the test tickets this router labelled."""
    # TODO 2: accuracy with its CI, macro-F1, Spanish accuracy, n, latency, cost and privacy for this router.
    return {"router": name}

In [ ]:
from mastery.finetune import SYSTEM_PROMPT

# 🖥️ The routers: numbers come from files recorded in 14.03/14.04 (yours in data/outputs/m14/ when present).
lr = LogReg().fit([t["text"] for t in TRAIN], [t["queue"] for t in TRAIN])
start = time.perf_counter()
lr_labels = {t["id"]: p for t, (p, _) in zip(TEST, lr.predict([t["text"] for t in TEST]))}
SOURCES = {"logistic regression (numpy)": (lr_labels, round((time.perf_counter() - start) / len(TEST), 4), 0.0, True)}
nb = NaiveBayes().fit([t["text"] for t in TRAIN], [t["queue"] for t in TRAIN])
start = time.perf_counter()
nb_labels = {t["id"]: p for t, (p, _) in zip(TEST, nb.predict([t["text"] for t in TEST]))}
SOURCES["naive Bayes (numpy)"] = (nb_labels, round((time.perf_counter() - start) / len(TEST), 4), 0.0, True)

for file in ["fewshot-qwen3-0.6b.json", "fewshot-qwen3-4b.json", "lora-qwen3-0.6b-cpu.json"]:
    rec = load_predictions(RECORDED_DIR / file)
    secs = rec.get("seconds") or rec.get("infer_seconds")
    SOURCES[rec["model"]] = (rec["labels"], round(secs / len(rec["labels"]), 3), 0.0, True)

groq_file, qlora_file = OUT_DIR / "fewshot-groq.json", OUT_DIR / "qlora-predictions.json"
if groq_file.exists():                                   # from Lesson 14.03's live cell (check #59)
    rec = load_predictions(groq_file)
    usd = estimate_cost({"input_tokens": rec["input_tokens"], "output_tokens": rec["output_tokens"]}, rec["model"]) or 0.0
    SOURCES[rec["model"]] = (rec["labels"], round(rec["seconds"] / len(rec["labels"]), 3), usd / len(rec["labels"]) * 1000, False)
else:
    print("(no fewshot-groq.json yet: run Lesson 14.03's 💸 cell to add the hosted few-shot router)")

TUNED = "orbita-router"
served = None
if qlora_file.exists():                                  # from the Colab notebook (Lesson 14.04)
    rec = load_predictions(qlora_file)
    if HAVE_OLLAMA and OLLAMA.get(TUNED):              # 🖥️ measure the served GGUF on 40 tickets for its laptop latency
        model = get_model(model=f"ollama:{TUNED}", temperature=0, max_tokens=8)
        start = time.perf_counter()
        for t in TEST[:40]:
            model.invoke([("system", SYSTEM_PROMPT), ("user", t["text"])])
        served = round((time.perf_counter() - start) / 40, 3)
    SOURCES[rec["model"]] = (rec["labels"], served, 0.0, True)
    print(f"Colab QLoRA run found; laptop latency: {served or 'not measured (serve it as orbita-router, Lesson 14.04 Part 4)'}")
else:
    print("(no qlora-predictions.json yet: run the Colab notebook of Lesson 14.04 to add your fine-tuned router)")

ROUTER_ROWS = [router_row(name, *src) for name, src in SOURCES.items()]
table(ROUTER_ROWS, ["router", "n", "accuracy [95% CI]", "macro-F1", "Spanish", "s/ticket", "USD/1K tickets", "local"])

**You should now see** (solution) at least five routers on the 234 tickets: logistic regression ≈ 63% and naive Bayes ≈ 60% in microseconds, few-shot qwen3:0.6b ≈ 47%, few-shot qwen3:4b ≈ 94% at ≈ 1.6 s per ticket, the course's LoRA-tuned 0.6B ≈ 87% at ≈ 0.1 s per ticket, plus your Groq (100 tickets) and Colab rows when their files exist.

**TODO 3** is in the cell below.

> 💡 **Hint:** For each pair, keep the test ids **both** routers labelled, turn each router's labels into 0/1 with `correct_map(labels, shared_gold)`, and call `paired_compare(a_scores, b_scores, names=(a, b))`. Keep `n`, wins, the difference, `p_value`, `significant` and `verdict()`.

<details>
<summary>Still stuck? Show one way to write it</summary>

```python
def paired_rows(pairs: list) -> list:
    """For each (a, b) router-name pair: paired comparison on the tickets both labelled."""
    out = []
    for a, b in pairs:
        la, lb = SOURCES[a][0], SOURCES[b][0]
        shared = {i: GOLD[i] for i in la if i in lb and i in GOLD}
        r = paired_compare(correct_map(la, shared), correct_map(lb, shared), names=(a, b))
        out.append({"a": a, "b": b, "n": r.n, "b wins": r.b_wins, "a wins": r.a_wins,
                    "diff (b - a)": f"{r.mean_diff.estimate:+.1%}", "p": float(f"{r.p_value:.2g}"),
                    "significant": r.significant, "verdict": r.verdict()})
    return out
```

</details>

In [ ]:
def paired_rows(pairs: list) -> list:
    """For each (a, b) router-name pair: paired comparison on the tickets both labelled."""
    # TODO 3: score both routers 0/1 on their shared tickets and run paired_compare; one dict per pair.
    return []

In [ ]:
# 🧪 The comparisons the recommendation rests on.
tuned_names = [n for n in SOURCES if n.startswith(("lora:", "qlora:"))]
PAIRS = [("ollama:qwen3:0.6b", n) for n in tuned_names] + [("ollama:qwen3:4b", n) for n in tuned_names]
PAIRS += [("logistic regression (numpy)", "ollama:qwen3:4b")]
PAIRS += [(n, "ollama:qwen3:4b") for n in SOURCES if n.startswith("groq:")]
PAIR_ROWS = paired_rows(PAIRS)
if PAIR_ROWS:
    table(PAIR_ROWS, ["a", "b", "n", "b wins", "a wins", "diff (b - a)", "p", "significant"])
else:
    print("(no paired comparisons yet: TODO 3)")

**You should now see** (solution) the tuned 0.6B far better than its few-shot base, significantly worse than few-shot qwen3:4b, and qwen3:4b far better than the baseline.

---
## Step 4 · The decision

**TODO 4** is in the cell below.

> 💡 **Hint:** Compare on routers that labelled all 234 tickets. Keep the ones with `local` (when `require_local`) and a known `s/ticket` within `budget_s`; choose the highest `accuracy.estimate`. Then find the most accurate router overall and use the paired row between the two to say whether the gap is significant. Return the name, a reason that quotes accuracy, latency and the paired result, and `fine_tuning_worth_it`.

<details>
<summary>Still stuck? Show one way to write it</summary>

```python
def recommend(rows: list, pair_rows: list, *, require_local: bool = True, budget_s: float = LATENCY_BUDGET_S) -> dict:
    """{"router": name, "reason": text}: the router Orbita Cloud should deploy."""
    full = [r for r in rows if r["n"] == len(TEST)]                      # compare on the whole test set only
    ok = [r for r in full if (r["local"] or not require_local) and r["s/ticket"] is not None and r["s/ticket"] <= budget_s]
    if not ok:
        return {"router": None, "reason": f"no router meets the {budget_s} s budget{' locally' if require_local else ''}"}
    pick = max(ok, key=lambda r: (r["accuracy"].estimate, -r["s/ticket"]))
    best = max(full, key=lambda r: r["accuracy"].estimate)
    gap = next((p for p in pair_rows if {p["a"], p["b"]} == {pick["router"], best["router"]}), None)
    tuned = pick["router"].startswith(("lora:", "qlora:"))
    reason = (f"{pick['router']} is the most accurate router that keeps tickets on the machine and answers within "
              f"{budget_s} s ({pick['accuracy [95% CI]']}, {pick['s/ticket']} s/ticket). ")
    if pick is best:
        reason += "It is also the most accurate router overall."
    else:
        sig = "significantly" if gap and gap["significant"] else "not significantly"
        reason += (f"The best router overall, {best['router']} ({best['accuracy [95% CI]']}), is {sig} better "
                   f"(paired p = {gap['p'] if gap else 'n/a'}) but takes {best['s/ticket']} s/ticket, over the budget. ")
    reason += ("Fine-tuning was worth it: a small tuned model meets the budget that few-shot prompting only meets with a weak model."
               if tuned else "Fine-tuning was not needed under these constraints.")
    return {"router": pick["router"], "reason": reason, "fine_tuning_worth_it": tuned}
```

</details>

In [ ]:
def recommend(rows: list, pair_rows: list, *, require_local: bool = True, budget_s: float = LATENCY_BUDGET_S) -> dict:
    """{"router": name, "reason": text}: the router Orbita Cloud should deploy."""
    # TODO 4: keep routers that are local (if required) and within the latency budget; pick the most accurate,
    # and say whether fine-tuning was worth it compared with the best few-shot router.
    return {"router": None, "reason": "not decided"}

In [ ]:
DECISION = recommend(ROUTER_ROWS, PAIR_ROWS)
print("→", DECISION["router"])
print(DECISION["reason"])
print("\nWithout the latency budget:", recommend(ROUTER_ROWS, PAIR_ROWS, budget_s=60)["router"])

**You should now see** (solution) `→ lora:Qwen/Qwen3-0.6B (r=16, 2 epochs)` (or your Colab model, if it is served and more accurate), with a reason quoting its accuracy, its latency and the paired gap to qwen3:4b, and `Without the latency budget: ollama:qwen3:4b`. The constraint, not the leaderboard, decides.

---
## Step 5 · The report

**TODO 5** is in the cell below.

> 💡 **Hint:** Build one Markdown string with the six `##` sections named in the TODO. Turn `evidence["readers"]`, `evidence["routers"]` and `evidence["pairs"]` into tables, include the reader paired result and the decision with its reason, and say the data are **synthetic**. Save it to `path` and return it.

<details>
<summary>Still stuck? Show one way to write it</summary>

```python
def write_report(path, evidence: dict) -> str:
    """The Markdown report for Orbita Cloud's support lead; returns its text."""
    readers = "\n".join(f"| {r['reader']} | {r['right']} | {r['accuracy [95% CI]']} | {r['s/question']} | {r['USD/1K questions']} | "
                        f"{r['data leaves the machine']} |" for r in evidence["readers"])
    routers = "\n".join(f"| {r['router']} | {r['n']} | {r['accuracy [95% CI]']} | {r['macro-F1']} | {r['Spanish']} | "
                        f"{r['s/ticket'] if r['s/ticket'] is not None else 'not measured'} | {r['USD/1K tickets']} | "
                        f"{'yes' if r['local'] else 'no'} |" for r in evidence["routers"])
    pairs = "\n".join(f"| {p['b']} vs {p['a']} | {p['n']} | {p['b wins']}–{p['a wins']} | {p['diff (b - a)']} | {p['p']} | "
                      f"{'yes' if p['significant'] else 'no'} |" for p in evidence["pairs"])
    d = evidence["decision"]
    text = f"""# Project 14 report: a local-first support assistant and ticket router

## Assistant

16 golden questions from the Project 06 test set (12 answerable, 4 not), same BM25 context for every reader.

| Reader | Right | Accuracy [95% CI] | s/question | USD/1K questions | Data leaves the machine |
|---|---|---|---|---|---|
{readers}

{evidence["reader_pair"]}

## Router

234 held-out test tickets (split by template family, never used for training or few-shot examples).

| Router | n | Accuracy [95% CI] | Macro-F1 | Spanish | s/ticket | USD/1K tickets | Local |
|---|---|---|---|---|---|---|---|
{routers}

## Comparisons

| Pair | n | Wins–losses | Difference | p | Significant |
|---|---|---|---|---|---|
{pairs}

## Recommendation

**{d["router"]}.** {d["reason"]}

## Privacy and costs

Local routers and readers keep tickets on the support laptop; hosted ones send the text to Groq (free tier: data
handling per Groq's terms, rate limits). Local cost is CPU time, not tokens; hosted cost uses paid-tier list prices.
Recorded runs were measured on an 8-core CPU without a GPU; the tickets and questions are **synthetic**.
"""
    path.write_text(text, encoding="utf-8")
    return text
```

</details>

In [ ]:
def write_report(path, evidence: dict) -> str:
    """The Markdown report for Orbita Cloud's support lead; returns its text."""
    # TODO 5: sections Assistant, Router, Comparisons, Recommendation, Privacy and costs; measured numbers only.
    text = "# Project 14 report\n\n(TODO)\n"
    path.write_text(text, encoding="utf-8")
    return text

In [ ]:
evidence = {"readers": reader_rows, "reader_pair": str(reader_pair), "routers": ROUTER_ROWS, "pairs": PAIR_ROWS, "decision": DECISION}
report = write_report(OUT_DIR / "project14-report.md", evidence)
print(report[:1200])

**You should now see** the report starting `# Project 14 report` (solution). Saved to `data/outputs/m14/project14-report.md`.

---
## Acceptance checks

In [ ]:
def accept(name, ok):
    results[name] = bool(ok)
    print(("✅" if ok else "❌"), name)


results = {}
reference = {}                                        # an independent scorer for check 1
for name, r in READERS.items():
    reference[name] = {q["id"]: int((q["answer_contains"].casefold() in (r["answers"].get(q["id"]) or "").casefold())
                                    if q["in_corpus"] else bool(DECLINE.search(r["answers"].get(q["id"]) or "")))
                       for q in GOLDEN}
accept("readers scored correctly on the 16 golden questions", SCORES == reference and sum(SCORES["ollama:qwen3:4b (recorded)"].values()) > 8)
accept("every reader answered the same 16 questions, local ones marked as private",
       all(len(r["answers"]) == len(GOLDEN) for r in READERS.values()) and READERS["ollama:qwen3:4b (recorded)"]["local"])
accept("router table: ≥ 5 routers on the full test set with CIs",
       sum(1 for r in ROUTER_ROWS if r.get("n") == len(TEST) and "accuracy [95% CI]" in r) >= 5)
accept("latency, cost and privacy for every router, local ones at $0",
       all({"s/ticket", "USD/1K tickets", "local"} <= set(r) for r in ROUTER_ROWS)
       and all(r["USD/1K tickets"] == 0 for r in ROUTER_ROWS if r.get("local")))
accept("paired tests: each tuned model vs its few-shot base and vs few-shot qwen3:4b",
       all(any(p["b"] == n and p["a"] == base for p in PAIR_ROWS) for n in tuned_names
           for base in ("ollama:qwen3:0.6b", "ollama:qwen3:4b")) and all("p" in p for p in PAIR_ROWS))
best_local_fast = max((r for r in ROUTER_ROWS if r.get("local") and r.get("n") == len(TEST) and r.get("s/ticket") is not None
                       and r["s/ticket"] <= LATENCY_BUDGET_S), key=lambda r: r["accuracy"].estimate, default=None)
overall = max((r for r in ROUTER_ROWS if r.get("n") == len(TEST)), key=lambda r: r["accuracy"].estimate, default=None)
accept("recommendation: the most accurate local router within the budget, with the fine-tuning verdict and paired evidence",
       best_local_fast is not None and DECISION.get("router") == best_local_fast["router"] and "fine_tuning_worth_it" in DECISION
       and ("paired" in DECISION.get("reason", "") or best_local_fast is overall))
from mastery.finetune import family_overlap
accept("held-out test set: 234 tickets, no template family shared with training", len(TEST) == 234 and not family_overlap(TRAIN, TEST))
accept("report: six sections with measured numbers and the synthetic-data note",
       all(s in report for s in ("## Assistant", "## Router", "## Comparisons", "## Recommendation", "## Privacy"))
       and "synthetic" in report and str(DECISION.get("router")) in report)
print(f"\n{sum(results.values())}/{len(results)} acceptance checks passed")
(OUT_DIR / "project14-acceptance.json").write_text(json.dumps(results, indent=2), encoding="utf-8")

**You should now see** `8/8 acceptance checks passed` (solution). Starter: `2/8` (the readers' shared questions and the held-out split come from the course; everything else needs a TODO). Results: `data/outputs/m14/project14-acceptance.json`.

## Findings (write yours)

- Which local reader would you give the support team today, and what would you fix first (retrieval or the reader)?
- With your own laptop's numbers, does the recommendation change? Rerun with `LATENCY_BUDGET_S = 2.0`.
- What would make you retrain the router (Module 12 monitoring), and how would you know?

## ✅ Before you commit

- [ ] Restart & Run All passes without a key and without Ollama.
- [ ] All TODOs done; 8/8 acceptance checks; report and findings written.
- [ ] Outputs cleared (`jupyter nbconvert --clear-output --inplace <notebook>`); `data/outputs/` not committed.

---
## ➡️ What's next

🎉 **Module 14 done:** you can run, choose, fine-tune and serve open models, and decide with evidence when it's worth it.

**Next:** the capstones in [COURSE_OUTLINE.md](../../../COURSE_OUTLINE.md). Back to the [Module 14 README](../../../modules/14-local-models-and-fine-tuning/README.md).